Tugas6_2505060019_Syifa Rahma Rasendriya

In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [40]:
# A. Extract
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_review = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("jumlah baris data transaksi: ", df_transaksi.count())
print("jumlah baris data produk: ", df_produk.count())
print("jumlah baris data review: ", df_review.count())

df_transaksi.printSchema()
df_produk.printSchema()
df_review.printSchema()

jumlah baris data transaksi:  5000
jumlah baris data produk:  30
jumlah baris data review:  3500
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



B. Transform - Penggabungan

In [41]:
# 1. Gabungkan transaksi ke ulasan
df_transaksi_ulasan = df_transaksi.join(df_review, on="order_id", how="left")
df_transaksi_ulasan.show(10)
print("Jumlah transaksi setelah join ulasan", df_transaksi_ulasan.count())

+--------+----------+------------+-------------------+------+
|order_id|product_id|unit_terjual|            tanggal|rating|
+--------+----------+------------+-------------------+------+
|     TX0|        21|           2|2026-10-10 00:00:00|     1|
|     TX1|         8|           6|2026-10-25 00:00:00|  NULL|
|     TX2|        25|           4|2026-10-13 00:00:00|     2|
|     TX3|         3|           4|2026-10-18 00:00:00|     5|
|     TX4|        19|           6|2026-10-07 00:00:00|  NULL|
|     TX5|        10|           3|2026-10-01 00:00:00|     5|
|     TX6|        26|           6|2026-10-24 00:00:00|     4|
|     TX7|         4|           1|2026-10-03 00:00:00|  NULL|
|     TX8|         7|           2|2026-10-09 00:00:00|     5|
|     TX9|        30|           1|2026-10-27 00:00:00|  NULL|
+--------+----------+------------+-------------------+------+
only showing top 10 rows

Jumlah transaksi setelah join ulasan 5000


In [42]:
# 2. Gabungkan dengan data Produk
df_etl = df_transaksi_ulasan.join(df_produk, on="product_id", how="inner")
df_etl.show(10)
df_etl.printSchema()

+----------+--------+------------+-------------------+------+------+------------+-----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|
+----------+--------+------------+-------------------+------+------+------------+-----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|
|        26|     TX6|           6|2026-10-24 00:00:00|     4| 75000|   Kesehatan|  Produk-26|
|         4|     TX7|           1|2026-10-03 00:00:00|  NULL

In [43]:
# 3. Menambahkan kollom total pendapatan
df_etl = df_etl.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
df_etl.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

In [44]:
# C. Transform - Penanganan Data Kosong & Pengayaan
df_etl = df_etl.withColumn("ada_ulasan", col("rating").isNotNull())
df_etl.show(10)

df_etl = df_etl.na.fill({"rating": 0})
df_etl.show(10)

print("Jumlah rating yang masih kosong: ", df_etl.filter(col("rating").isNull()).count())

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|     false|
|        10|     TX5|   

In [45]:
# D. Load
!hdfs dfs -mkdir -p /user/mahasiswa/tugas6/hasil_etl
df_etl.write.mode("overwrite").partitionBy("kategori").parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")
print("Data berhasil disimpan di HDFS.")

!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl
df_hdfs = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl")
df_hdfs.show(5)

[Stage 36:>                                                         (0 + 1) / 1]

Data berhasil disimpan di HDFS.


-rw-r--r--   3 syifa supergroup          0 2026-09-24 05:06 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - syifa supergroup          0 2026-09-24 05:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 syifa supergroup      16363 2026-09-24 05:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-088d9dd8-f49c-4316-a56c-ded817e545a9.c000.snappy.parquet
drwxr-xr-x   - syifa supergroup          0 2026-09-24 05:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 syifa supergroup      11130 2026-09-24 05:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-088d9dd8-f49c-4316-a56c-ded817e545a9.c000.snappy.parquet
drwxr-xr-x   - syifa supergroup          0 2026-09-24 05:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 syifa supergroup      10388 2026-09-24 05:06 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-088d9dd8-f49c-4316-a56c-ded817e545a9.c000.snappy.parquet
drwxr-xr-x   - syifa sup

In [46]:
# E. Insight Akhir
from pyspark.sql.functions import col,count,sum as spark_sum, when
df_insight = df_hdfs.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"), 
    spark_sum(
        when(col("ada_ulasan") == True,1).otherwise(0)
    ).alias("jumlah_dengan_ulasan")
)

df_insight = df_insight.withColumn(
    "persentase_ulasan", 
    col("jumlah_dengan_ulasan") / col("total_transaksi") * 100)
df_insight.orderBy("persentase_ulasan").show()

+------------+---------------+--------------------+-----------------+
|    kategori|total_transaksi|jumlah_dengan_ulasan|persentase_ulasan|
+------------+---------------+--------------------+-----------------+
|     Makanan|            536|                 369|68.84328358208955|
|   Kesehatan|            961|                 662|68.88657648283039|
|     Fashion|           1035|                 726|70.14492753623188|
|Rumah Tangga|            815|                 575| 70.5521472392638|
|  Elektronik|           1653|                1168|  70.659407138536|
+------------+---------------+--------------------+-----------------+



Mengapa hal ini mungkin penting diketahui oleh tim marketing?

Kategori yang memiliki persentase transaksi dengan ulasan rendah menunjukkan bahwa sebagian besar transaksi pada kategori tersebut belum 
memberikan ulasan. Informasi ini dapat digunakan oleh tim marketing untuk mengetahui kategori yang masih mempunyai sedikit feedback dari 
pelanggan. Sehingga dapat menjadi bahan pertimbangan dalam membuat strategi untuk meningkatkan jumlah ulasan pelanggan.

In [47]:
# EKSPLORASI
# Menghapus kolom 
df_ringkas = df_transaksi.drop("rating", "kategori")

df_ringkas.show()
df_ringkas.printSchema()

+--------+----------+------------+-------------------+
|order_id|product_id|unit_terjual|            tanggal|
+--------+----------+------------+-------------------+
|     TX0|        21|           2|2026-10-10 00:00:00|
|     TX1|         8|           6|2026-10-25 00:00:00|
|     TX2|        25|           4|2026-10-13 00:00:00|
|     TX3|         3|           4|2026-10-18 00:00:00|
|     TX4|        19|           6|2026-10-07 00:00:00|
|     TX5|        10|           3|2026-10-01 00:00:00|
|     TX6|        26|           6|2026-10-24 00:00:00|
|     TX7|         4|           1|2026-10-03 00:00:00|
|     TX8|         7|           2|2026-10-09 00:00:00|
|     TX9|        30|           1|2026-10-27 00:00:00|
|    TX10|        20|           1|2026-10-04 00:00:00|
|    TX11|        15|           4|2026-10-01 00:00:00|
|    TX12|        29|           3|2026-10-02 00:00:00|
|    TX13|        11|           7|2026-10-24 00:00:00|
|    TX14|        14|           1|2026-10-01 00:00:00|
|    TX15|

In [48]:
# Mengurutkan transaksi dari pendapatan tertinggi ke terendah
df_terurut = df_etl.orderBy(col("total_pendapatan").desc())

# Tampilkan 5 transaksi tertinggi
df_terurut.select("order_id", "kategori", "total_pendapatan").show(5)

+--------+------------+----------------+
|order_id|    kategori|total_pendapatan|
+--------+------------+----------------+
|   TX135|     Fashion|         1750000|
|   TX313|     Fashion|         1750000|
|   TX154|  Elektronik|         1750000|
|    TX44|Rumah Tangga|         1750000|
|   TX197|     Fashion|         1750000|
+--------+------------+----------------+
only showing top 5 rows



In [49]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
